# Animagine XL 4.0 — Kaggle GPU

ตั้งค่า Kaggle Notebook ให้เปิด **Internet** และเลือก **GPU T4 x2** (หรือ GPU ที่มี) ก่อนเลือก Run All ครับ โมเดลโหลดจาก Hugging Face โดยตรง ไม่ต้องตั้ง HF_TOKEN สำหรับ repo สาธารณะนี้

> หากติดตั้งแพ็กเกจแล้วเจอปัญหา import ให้เลือก Runtime → Restart session แล้วรัน Run All อีกครั้ง

In [ ]:
# ติดตั้งส่วนเสริมโดยไม่ถอน gym และไม่อัปเกรด PyTorch/CUDA ที่ Kaggle เตรียมไว้
%pip install -q -U diffusers transformers accelerate safetensors
%pip install -q 'gradio>=4.44,<5'

import random
import torch
import gradio as gr
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

if not torch.cuda.is_available():
    raise RuntimeError('ไม่พบ GPU: เปิด Notebook Settings แล้วเลือก Accelerator เป็น GPU')

gpu_count = torch.cuda.device_count()
print(f'พบ GPU {gpu_count} ตัว: ' + ', '.join(torch.cuda.get_device_name(i) for i in range(gpu_count)))
print('หมายเหตุ: VRAM ของ GPU สองตัวไม่รวมกัน; notebook นี้ใช้ GPU 0 พร้���ม CPU offload')

# ชื่อ repo ที่ถูกต้อง (ไม่ใช่ cagliostrolab/animagine-xl-4.0-opt)
model_id = 'cagliostrolab/animagine-xl-4.0'

try:
    pipe = StableDiffusionXLPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        use_safetensors=True,
        custom_pipeline='lpw_stable_diffusion_xl',
        add_watermarker=False,
)
except Exception as exc:
    raise RuntimeError(
        f'โหลด {model_id} ไม่สำเร็จ: ตรวจว่า Kaggle เปิด Internet แล้ว และ Hugging Face เข้าถึงได้ ' \
        'จากนั้นรัน cell นี้ใหม่ หากเป็น 401/403 ให้เปิดหน้าโมเดลและยอมรับเงื่อนไขการใช้งานก่อน'
    ) from exc

pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
pipe.enable_model_cpu_offload(gpu_id=0)
print('โหลดโมเดลสำเร็จ พร้อมสร้างภาพ')

def generate_anime_v4(prompt, negative_prompt, aspect_ratio, steps, cfg_scale, seed):
    sizes = {
        'จัตุรัส (1:1)': (1024, 1024),
        'แนวตั้งมาตรฐาน (3:4)': (896, 1152),
        'แนวตั้ง (9:16 - มือถือ)': (832, 1216),
        'แนวนอน (16:9 - หน้าจอคอม)': (1216, 832),
    }
    width, height = sizes.get(aspect_ratio, (896, 1152))
    actual_seed = random.randint(0, 2147483647) if seed is None or int(seed) == -1 else int(seed)
    generator = torch.Generator(device='cpu').manual_seed(actual_seed)
    try:
        with torch.inference_mode():
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                width=width,
                height=height,
                num_inference_steps=int(steps),
                guidance_scale=float(cfg_scale),
                generator=generator,
            )
        return result.images[0], actual_seed
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        raise gr.Error('หน่วยความจำ GPU ไม่พอ: ลองเลือกภาพขนาดเล็กลงหรือลด Steps') from exc

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🌸 Animagine XL 4.0 — สร้างภาพอนิเมะ')
    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label='Prompt', lines=3,
                value='1girl, silver hair, blue eyes, stylish casual clothes, sunset city background, masterpiece, high score, great score, absurdres',
            )
            neg_prompt_input = gr.Textbox(
                label='Negative Prompt', lines=3,
                value='lowres, bad anatomy, bad hands, text, error, missing fingers, extra digits, fewer digits, cropped, worst quality, low quality, low score, bad score, average score',
            )
            aspect_dropdown = gr.Dropdown(
                choices=['จัตุรัส (1:1)', 'แนวตั้งมาตรฐาน (3:4)', 'แนวตั้ง (9:16 - มือถือ)', 'แนวนอน (16:9 - หน้าจอคอม)'],
                value='แนวตั้งมาตรฐาน (3:4)', label='สัดส่วนภาพ',
            )
            with gr.Row():
                steps_slider = gr.Slider(15, 40, value=28, step=1, label='Steps')
                cfg_slider = gr.Slider(1.0, 10.0, value=5.0, step=0.5, label='CFG Scale')
            seed_input = gr.Number(value=-1, label='Seed (-1 = สุ่ม)', precision=0)
            generate_btn = gr.Button('✨ สร้างภาพ', variant='primary')
        with gr.Column():
            output_image = gr.Image(label='ผลลัพธ์')
            seed_output = gr.Number(label='Seed ที่ใช้', interactive=False)
    generate_btn.click(
        fn=generate_anime_v4,
        inputs=[prompt_input, neg_prompt_input, aspect_dropdown, steps_slider, cfg_slider, seed_input],
        outputs=[output_image, seed_output],
    )

demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)